In [ ]:
# ============================================================
#  NOTEBOOK 5 — Launch Flask Web App  (1 Cell)
# ============================================================
import subprocess, sys, time, os, threading, urllib.request, urllib.error

# ── Install dependencies ──────────────────────────────────────────────────────
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install",
                "flask", "--quiet"], check=False)
print("✅ Dependencies ready")

# ── Verify app.py ─────────────────────────────────────────────────────────────
if not os.path.exists("app.py"):
    raise FileNotFoundError(
        "\n❌ app.py not found!\n"
        "Save app.py in the SAME folder as this notebook."
    )
print("✅ app.py found")

# ── Verify model files ────────────────────────────────────────────────────────
REQUIRED    = ["models/scaler_X.pkl", "models/scaler_y.pkl",
               "data/cpi_data.csv"]
MODEL_NAMES = ["lstm_model", "gru_model", "bilstm_model",
               "cnn_lstm_model", "transformer_model"]

missing = [f for f in REQUIRED if not os.path.exists(f)]
for name in MODEL_NAMES:
    if not os.path.exists(f"models/{name}.keras") and \
       not os.path.exists(f"models/{name}.h5"):
        missing.append(f"models/{name}.keras  or  .h5")

if missing:
    print("\n❌ Missing files — run notebooks 01 → 04 first:")
    for m in missing: print("   •", m)
    raise SystemExit("Fix missing files then re-run.")
print("✅ All required files found")

# ── Kill any process on port 5000 ─────────────────────────────────────────────
try:
    if sys.platform == "win32":
        r = subprocess.run(["netstat", "-ano"],
                           capture_output=True, text=True)
        for line in r.stdout.splitlines():
            if ":5000" in line and "LISTENING" in line:
                pid = line.strip().split()[-1]
                subprocess.run(["taskkill", "/F", "/PID", pid],
                                capture_output=True)
                print(f"⚠️  Killed existing process on port 5000 (PID {pid})")
                break
    else:
        subprocess.run(["fuser", "-k", "5000/tcp"],
                       capture_output=True, check=False)
except Exception as e:
    print(f"Port cleanup note: {e}")

time.sleep(1)

# ── Launch Flask ──────────────────────────────────────────────────────────────
log_lines = []

def stream_logs(proc):
    for line in proc.stdout:
        txt = line.decode("utf-8", errors="replace").rstrip()
        log_lines.append(txt)
        print("[Flask]", txt)

flask_proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd=os.getcwd()
)

threading.Thread(target=stream_logs,
                 args=(flask_proc,), daemon=True).start()

# ── Wait until Flask responds ─────────────────────────────────────────────────
print("\n⏳ Waiting for Flask", end="", flush=True)
started = False
for _ in range(40):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/", timeout=1)
        started = True
        break
    except Exception:
        pass

print()
if started:
    print("\n" + "="*50)
    print("  ✅ Flask is live!")
    print("  🌐 Open → http://localhost:5000")
    print("="*50)
    try:
        import webbrowser
        webbrowser.open("http://localhost:5000")
    except Exception:
        pass
else:
    print("\n❌ Flask did not start. Last log output:")
    for line in log_lines[-20:]:
        print("   ", line)
    print("\n── Run manually in a terminal ──")
    print(f"  cd \"{os.getcwd()}\"")
    print("  python app.py")

print("\n💡 To stop the server: run  flask_proc.terminate()  in a new cell")